<a href="https://colab.research.google.com/github/iescobedo5/Computer-Vision/blob/main/SSD10k_BYTE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q supervision

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 7.1 MB/s eta 0:00:00


In [ ]:
import torch
from torchvision.models.detection import ssd300_vgg16
from torchvision.models.detection.ssd import SSDClassificationHead

WEIGHTS_PATH = "/content/drive/MyDrive/UA/ssd_results/subset10k_25e/ssd_best_10k25e.pth"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

num_classes = 2  # background + vehicle

model = ssd300_vgg16(weights=None)

in_channels = [512, 1024, 512, 256, 256, 256]
num_anchors = model.anchor_generator.num_anchors_per_location()

model.head.classification_head = SSDClassificationHead(
    in_channels=in_channels,
    num_anchors=num_anchors,
    num_classes=num_classes
)

model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device))
model.to(device)
model.eval()

SSD(
  (backbone): SSDFeatureExtractorVGG(
    (features): Sequential(
      (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU(inplace=True)
      (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU(inplace=True)
      (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (6): ReLU(inplace=True)
      (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (8): ReLU(inplace=True)
      (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (11): ReLU(inplace=True)
      (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (13): ReLU(inplace=True)
      (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (15): ReLU(inplace=

In [ ]:
import torch
import supervision as sv
import numpy as np
from torchvision.transforms import functional as F
from PIL import Image, ImageDraw
from pathlib import Path
import time

IMAGE_DIR = Path("/content/drive/MyDrive/MVI_39031")
SAVE_DIR = Path("/content/drive/MyDrive/UA/ssd10k_bytetrack/MVI_39031")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

TRACK_TXT = SAVE_DIR / "ssd10k_bytetrack_pred.txt"

CONF_THRESH = 0.25

tracker = sv.ByteTrack(
    track_activation_threshold=0.25,
    lost_track_buffer=30,
    minimum_matching_threshold=0.8,
    frame_rate=25
)

image_files = sorted([
    p for p in IMAGE_DIR.glob("*")
    if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
])

track_lines = []
start = time.time()

with torch.no_grad():
    for frame_idx, img_path in enumerate(image_files, start=1):
        img = Image.open(img_path).convert("RGB")
        img_tensor = F.to_tensor(img).to(device)

        outputs = model([img_tensor])[0]

        boxes = outputs["boxes"].cpu().numpy()
        scores = outputs["scores"].cpu().numpy()
        labels = outputs["labels"].cpu().numpy()

        keep = scores >= CONF_THRESH

        detections = sv.Detections(
            xyxy=boxes[keep],
            confidence=scores[keep],
            class_id=labels[keep]
        )

        tracked_detections = tracker.update_with_detections(detections)

        draw = ImageDraw.Draw(img)

        for xyxy, conf, class_id, track_id in zip(
            tracked_detections.xyxy,
            tracked_detections.confidence,
            tracked_detections.class_id,
            tracked_detections.tracker_id
        ):
            x1, y1, x2, y2 = xyxy
            w = x2 - x1
            h = y2 - y1

            draw.rectangle([x1, y1, x2, y2], outline="red", width=2)
            draw.text((x1, y1), f"ID {track_id} {conf:.2f}", fill="red")

            # TrackEval/MOT format:
            # frame, id, x, y, w, h, confidence, -1, -1, -1
            track_lines.append(
                f"{frame_idx},{int(track_id)},{x1:.2f},{y1:.2f},{w:.2f},{h:.2f},{conf:.4f},-1,-1,-1\n"
            )

        img.save(SAVE_DIR / img_path.name)

end = time.time()
fps = len(image_files) / (end - start)

with open(TRACK_TXT, "w") as f:
    f.writelines(track_lines)

print("SSD + ByteTrack complete!")
print(f"Saved tracking file to: {TRACK_TXT}")
print(f"Saved annotated images to: {SAVE_DIR}")
print(f"Total tracked boxes: {len(track_lines)}")
print(f"FPS: {fps:.2f}")

SSD + ByteTrack complete!
Saved tracking file to: /content/drive/MyDrive/UA/ssd10k_bytetrack/MVI_39031/ssd10k_bytetrack_pred.txt
Saved annotated images to: /content/drive/MyDrive/UA/ssd10k_bytetrack/MVI_39031
Total tracked boxes: 6106
FPS: 3.05


step above took 8 min

In [ ]:
!rm -rf /content/TrackEval
!git clone https://github.com/JonathonLuiten/TrackEval.git
%cd /content/TrackEval
!pip install scipy matplotlib tqdm

Cloning into 'TrackEval'...
remote: Enumerating objects: 924, done.
remote: Counting objects: 100% (375/375), done.
remote: Compressing objects: 100% (60/60), done.
remote: Total 924 (delta 340), reused 315 (delta 315), pack-reused 549 (from 1)
Receiving objects: 100% (924/924), 393.80 KiB | 12.70 MiB/s, done.
Resolving deltas: 100% (619/619), done.
/content/TrackEval


In [ ]:
# add ground truth file to TrackEval
!mkdir -p /content/TrackEval/data/gt/mot_challenge/UA-DETRAC-train/MVI_39031/gt
!cp /content/drive/MyDrive/UA_DETRAC_GT/gt.txt /content/TrackEval/data/gt/mot_challenge/UA-DETRAC-train/MVI_39031/gt/gt.txt

In [ ]:
# add prediction file to TrackEval
!mkdir -p /content/TrackEval/data/trackers/mot_challenge/UA-DETRAC-train/ssd10k_bytetrack/data
!cp /content/drive/MyDrive/UA/ssd10k_bytetrack/MVI_39031/ssd10k_bytetrack_pred.txt /content/TrackEval/data/trackers/mot_challenge/UA-DETRAC-train/ssd10k_bytetrack/data/MVI_39031.txt

In [ ]:
from pathlib import Path

seqmap_dir = Path("/content/TrackEval/data/gt/mot_challenge/seqmaps")
seqmap_dir.mkdir(parents=True, exist_ok=True)

seqmap_file = seqmap_dir / "UA-DETRAC-train.txt"

seqmap_file.write_text("name\nMVI_39031\n")

print(seqmap_file.read_text())

name
MVI_39031



In [ ]:
import numpy as np
from pathlib import Path

gt_path = Path("/content/TrackEval/data/gt/mot_challenge/UA-DETRAC-train/MVI_39031/gt/gt.txt")
pred_path = Path("/content/TrackEval/data/trackers/mot_challenge/UA-DETRAC-train/ssd10k_bytetrack/data/MVI_39031.txt")

gt = np.loadtxt(gt_path, delimiter=",")
pred = np.loadtxt(pred_path, delimiter=",")

gt_max_frame = int(gt[:, 0].max())

# Handle case where pred is empty
if pred.size > 0:
    pred_max_frame = int(pred[:, 0].max())
else:
    pred_max_frame = 0 # If no predictions, max frame is 0

print("GT max frame:", gt_max_frame)
print("Prediction max frame:", pred_max_frame)

seq_length = max(gt_max_frame, pred_max_frame)

seqinfo_path = Path("/content/TrackEval/data/gt/mot_challenge/UA-DETRAC-train/MVI_39031/seqinfo.ini")

seqinfo_text = f"""
[Sequence]
name=MVI_39031
imDir=img1
frameRate=25
seqLength={seq_length}
imWidth=960
imHeight=540
imExt=.jpg
"""

seqinfo_path.write_text(seqinfo_text.strip())

print(seqinfo_path.read_text())

GT max frame: 1470
Prediction max frame: 1470
[Sequence]
name=MVI_39031
imDir=img1
frameRate=25
seqLength=1470
imWidth=960
imHeight=540
imExt=.jpg


In [ ]:
%cd /content/TrackEval

!grep -R "np.float" -n trackeval
!find trackeval -type f -name "*.py" -exec sed -i 's/np.float/float/g' {} +

/content/TrackEval
trackeval/metrics/hota.py:31:            res[field] = np.zeros((len(self.array_labels)), dtype=np.float)
trackeval/metrics/hota.py:37:            res['HOTA_FN'] = data['num_gt_dets'] * np.ones((len(self.array_labels)), dtype=np.float)
trackeval/metrics/hota.py:38:            res['LocA'] = np.ones((len(self.array_labels)), dtype=np.float)
trackeval/metrics/hota.py:42:            res['HOTA_FP'] = data['num_tracker_dets'] * np.ones((len(self.array_labels)), dtype=np.float)
trackeval/metrics/hota.py:43:            res['LocA'] = np.ones((len(self.array_labels)), dtype=np.float)
trackeval/metrics/track_map.py:223:            tp_sum = np.cumsum(tps, axis=1).astype(dtype=np.float)
trackeval/metrics/track_map.py:224:            fp_sum = np.cumsum(fps, axis=1).astype(dtype=np.float)
trackeval/metrics/track_map.py:262:            res['AP_' + lbl] = np.zeros((len(self.array_labels)), dtype=np.float)
trackeval/metrics/track_map.py:263:            res['AR_' + lbl] = np.zeros((len(

In [ ]:
%cd /content/TrackEval

!grep -R "np.int" -n trackeval
!find trackeval -type f -name "*.py" -exec sed -i 's/np.int/int/g' {} +

/content/TrackEval
trackeval/metrics/identity.py:83:        res['IDFN'] = fn_mat[match_rows, match_cols].sum().astype(np.int)
trackeval/metrics/identity.py:84:        res['IDFP'] = fp_mat[match_rows, match_cols].sum().astype(np.int)
trackeval/metrics/identity.py:85:        res['IDTP'] = (gt_id_count.sum() - res['IDFN']).astype(np.int)
trackeval/baselines/stp.py:52:                'ids': np.array([], np.int),
trackeval/baselines/stp.py:83:            ids = np.nan * np.ones((len(boxes),), np.int)
trackeval/datasets/kitti_2d_box.py:307:            to_remove_matched = np.array([], np.int)
trackeval/datasets/kitti_2d_box.py:365:                    data['gt_ids'][t] = gt_id_map[data['gt_ids'][t]].astype(np.int)
trackeval/datasets/kitti_2d_box.py:372:                    data['tracker_ids'][t] = tracker_id_map[data['tracker_ids'][t]].astype(np.int)
trackeval/datasets/rob_mots.py:446:                to_remove_tracker = np.array([], dtype=np.int)
trackeval/datasets/rob_mots.py:471:              

In [ ]:
!python scripts/run_mot_challenge.py \
  --BENCHMARK UA-DETRAC \
  --SPLIT_TO_EVAL train \
  --TRACKERS_TO_EVAL ssd10k_bytetrack \
  --METRICS HOTA CLEAR Identity \
  --USE_PARALLEL False


Eval Config:
USE_PARALLEL         : False                         
NUM_PARALLEL_CORES   : 8                             
BREAK_ON_ERROR       : True                          
RETURN_ON_ERROR      : False                         
LOG_ON_ERROR         : /content/TrackEval/error_log.txt
PRINT_RESULTS        : True                          
PRINT_ONLY_COMBINED  : False                         
PRINT_CONFIG         : True                          
TIME_PROGRESS        : True                          
DISPLAY_LESS_PROGRESS : False                         
OUTPUT_SUMMARY       : True                          
OUTPUT_EMPTY_CLASSES : True                          
OUTPUT_DETAILED      : True                          
PLOT_CURVES          : True                          

MotChallenge2DBox Config:
PRINT_CONFIG         : True                          
GT_FOLDER            : /content/TrackEval/data/gt/mot_challenge/
TRACKERS_FOLDER      : /content/TrackEval/data/trackers/mot_challenge/
OUTPUT_FOL